In [1]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

# Loading and diving into input and output
data = load_diabetes()
X, y = data.data, data.target

#Scaling for Gradient Descent to converge quickly.
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [2]:
import numpy as np

interactions = []
for i in range(X.shape[1]):
    for j in range(i + 1, X.shape[1]):
        new_col = X[:, i] * X[:, j]
        interactions.append(new_col)

X_prod = np.column_stack(interactions) # mistake 1: had done np.hstack instead of np.column_stack
interactions2 = []
for i in range(X.shape[1]):
    for j in range(i + 1, X.shape[1]):
        new_col = X[:, i]/ (X[:, j]+1e-7)
        interactions2.append(new_col)
X_div = np.column_stack(interactions)
X2 = X**2
X_poly = np.hstack([X,X2,X_prod,X_div])
X_poly_mean = np.mean(X_poly, axis = 0)
X_poly_std = np.std(X_poly, axis = 0)
X_poly = (X_poly - X_poly_mean)/X_poly_std

In [3]:
# Splitting into train, test and validation data
from sklearn.model_selection import train_test_split
train_X,test_X,train_y,test_y = train_test_split(X_poly,y,test_size = 0.2, random_state = 0)
train_X,val_X,train_y,val_y = train_test_split(train_X,train_y,test_size = 0.2, random_state = 0)
print(train_X.shape, test_X.shape, val_X.shape, train_y.shape, test_y.shape, val_y.shape)

(282, 110) (89, 110) (71, 110) (282,) (89,) (71,)


In [4]:
#train and validation phase
import numpy as np
w = np.zeros(train_X.shape[1],)
b = np.zeros(1,)
lr = 3e-3
J_best = 1e10
epochs = 1100
lambda1 = 1e-1
lambda2 = 1e-1
for epoch in range(epochs):
  # Train
  y = train_X@w + b
  J_train = np.mean((y-train_y)**2) + lambda1 * np.sum(np.absolute(w)) + lambda2* np.sum(w**2)
  dJdw = (1/train_X.shape[0])*(2*train_X.T@(y-train_y)) + lambda1 * np.sign(w) + 2*lambda2*w
  dJdb = np.mean(2*(y-train_y))
  w -= lr* dJdw
  b -= lr* dJdb
  # Validation
  y = val_X@w + b
  J_val = np.mean((y-val_y)**2) + lambda1 * np.sum(np.absolute(w))
  if (epoch+1)%100 == 0:
    print(f"{epoch+1}: train loss = {J_train}, validation loss = {J_val}")
  if J_val<J_best:
    J_best = J_val
    w_best = w.copy()
    b_best = b.copy()
print(f"Best validation loss = {J_best}")

100: train loss = 9599.9121363066, validation loss = 14191.163398804369
200: train loss = 4813.505214002972, validation loss = 7873.84278142376
300: train loss = 3284.8717641643534, validation loss = 5440.284574528295
400: train loss = 2767.962225780253, validation loss = 4456.9340822526265
500: train loss = 2584.146095973737, validation loss = 4034.04542398983
600: train loss = 2515.0504869737542, validation loss = 3839.4612107401313
700: train loss = 2487.173929727658, validation loss = 3745.983159698271
800: train loss = 2475.187316414341, validation loss = 3701.1042838767257
900: train loss = 2469.487144904584, validation loss = 3676.6957369916604
1000: train loss = 2466.454547441211, validation loss = 3664.3517848812476
1100: train loss = 2464.6554503539332, validation loss = 3657.6092787702846
Best validation loss = 3657.6092787702846


In [5]:
#testing phase
y = test_X@w + b
J_test = np.mean((y-test_y)**2)
print(J_test)

3252.9476455129616
